# **Heart_Disease_Identification_Unit(I)**

# 0. First import some formulas and functions we will be using!

In [ ]:
#@title
import pandas as pd
import numpy as np
import seaborn as sns
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import itertools

from keras.models import Model, Sequential
from keras.layers import Conv1D, GlobalAveragePooling1D, MaxPooling1D
from keras.layers import Activation, Dense, Dropout, Input, Embedding, Flatten
from keras.callbacks import CSVLogger
from tensorflow.keras.optimizers import RMSprop,Adam

from ast import literal_eval
from keras.utils.np_utils import to_categorical
import matplotlib.pyplot as plt
from keras.layers import LeakyReLU

# 1. First enter the ECG signal that group 7 peak into one set into the codebase!

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))

df = pd.read_csv('CNN_ECG_Signal.csv',delimiter=',',converters=dict(v2 = literal_eval ))
df.info()

# 2. Let's look at what the ECG signal looks like!

In [ ]:
#Read data
data = np.array(df)
data2 = data[0 : , 1]

#Config the figure setting!
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=data2[0], #Sample a section of normal ECG signal
    mode='lines',
    name='Original Plot',

)
)
fig.update_layout(
    title="A section of normal ECG signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig.show()

In [ ]:
fig2 = go.Figure()
fig2.add_trace(go.Scatter(
    y=data2[35], #Sample a section of arrhythmic ECG signal
    mode='lines',
    name='Original Plot'
))
fig2.update_layout(
    title="A section of arrhythmic ECG signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
#Display your result!
fig2.show()

# 3. [Pad zero] for these data to fix the data length!


*   Observe the data length above. Modify lengthOfData
*   Fine-tune lengthOfData until it is big enough to cause the resulting error of the program to be zero
*   lengthOfData that is too big will cause a heavy burden on the computer. Try to find the smallest lengthOfData!

In [ ]:
X = df.iloc[:,1:].values
lengthOfData = 3000 #An estimate of largest data length                              #Mysterious voice: Or you want to try 3000 ^^
numberOfData = 60

tempZ = np.zeros(shape=(numberOfData,lengthOfData))
tempX = []
error = 0
count=0

for a in X:
    amountOfPadding = lengthOfData - len(a[0])

    if len(a[0]) < lengthOfData :
        #If the data length is smaller than lengthOfData we will pad zero
        tempX = np.pad(a[0], (0,amountOfPadding), 'constant', constant_values=(0,0))
        tempZ[count] = tempX
        count = count + 1
    else:
        error = error + 1 #If the data length is larger than lengthOfData we will add an error count
        tempX = np.pad(a[0], (0,0), 'constant', constant_values=(0,0))
        count = count + 1


Y = df.iloc[:,:1].values
Y = to_categorical(Y,num_classes=2)

if error != 0 :
  print('error=',error, "Try to increase lengthOfData!")
else:
  print('error=',error, "Successfully pad zero, continue to the next step!")

# 4. Observe what this [zero padded] data looks like!



In [ ]:
#Config the figure setting!
fig3 = go.Figure()
fig3.add_trace(go.Scatter(
    y=tempZ[0], #Sample a section of normal ECG signal
    mode='lines',
    name='Original Plot',
)
)
fig3.update_layout(
    title="A section of [zero padded] normal ECG signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig3.show()

In [ ]:
#Config the figure setting!
fig4 = go.Figure()
fig4.add_trace(go.Scatter(
    y=tempZ[35], #Sample a section of arrhythmic ECG signal
    mode='lines',
    name='Original Plot',
)
)
fig4.update_layout(
    title="A section of [zero padded] arrhythmic ECG signal",
    font=dict(
        family="Courier New, monospace",
        size=20,
        color="RebeccaPurple"
    )
)
fig4.show()


# Unit Summary
---
1. Arrhythmic ECG signal appear much more irregular compare to normal ECG signal
2. ECG signal data doesn't have set pixel or format like pictures
3. Through zero padding technique we can set the format for our ECG signal